In [1]:
# Cell 1: Imports and paths
import cv2
import os
import numpy as np
import torch
from pathlib import Path
from deep_sort_realtime.deepsort_tracker import DeepSort
from ultralytics import YOLO
import pandas as pd
from scipy.optimize import linear_sum_assignment

# Base path
basePath = "./Object_Tracking/"
task1ImagesPath = os.path.join(basePath, "Task1/images")
task1GtPath = os.path.join(basePath, "Task1/gt/gt.txt")
task2ImagesPath = os.path.join(basePath, "Task2/images")

In [2]:
# Cell 2: Convert Task1 images to video
frameRate = 14
outputVideoPath = "task1_input.mp4"
images = sorted([img for img in os.listdir(task1ImagesPath) if img.endswith(".jpg")], key=lambda x: int(x.split('.')[0]))
sampleImage = cv2.imread(os.path.join(task1ImagesPath, images[0]))
height, width, _ = sampleImage.shape
videoWriter = cv2.VideoWriter(outputVideoPath, cv2.VideoWriter_fourcc(*'mp4v'), frameRate, (width, height))

for imgName in images:
    frame = cv2.imread(os.path.join(task1ImagesPath, imgName))
    videoWriter.write(frame)

videoWriter.release()

print(images[:20])


['000001.jpg', '000002.jpg', '000003.jpg', '000004.jpg', '000005.jpg', '000006.jpg', '000007.jpg', '000008.jpg', '000009.jpg', '000010.jpg', '000011.jpg', '000012.jpg', '000013.jpg', '000014.jpg', '000015.jpg', '000016.jpg', '000017.jpg', '000018.jpg', '000019.jpg', '000020.jpg']


In [3]:
# Cell 3: Load YOLOv8 model and DeepSORT tracker
yoloModel = YOLO("yolov8x.pt")  # Change to your trained model if needed
tracker = DeepSort(max_age=15, n_init=1)

c:\Users\peste\AppData\Local\Programs\Python\Python313\Lib\site-packages\deep_sort_realtime\embedder\embedder_pytorch.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [4]:
# Cell 4: Process video for Task1
cap = cv2.VideoCapture("task1_input.mp4")
outVideoPath = "task1.mp4"
videoWriter = cv2.VideoWriter(outVideoPath, cv2.VideoWriter_fourcc(*'mp4v'), frameRate, (width, height))
trackingResults = []

frameId = 1
while True:
    ret, frame = cap.read()
    if not ret:
        break
    results = yoloModel(frame)[0]
    boxes = results.boxes.xyxy.cpu().numpy()
    confidences = results.boxes.conf.cpu().numpy()
    classes = results.boxes.cls.cpu().numpy()

    detections = []
    for i, cls in enumerate(classes):
        if cls == 0:  # pedestrian class
            x1, y1, x2, y2 = boxes[i]
            detections.append(([x1, y1, x2-x1, y2-y1], confidences[i], "person"))

    tracks = tracker.update_tracks(detections, frame=frame)

    for track in tracks:
        if not track.is_confirmed():
            continue
        trackId = track.track_id
        l, t, w, h = track.to_ltrb()
        l, t, w, h = int(l), int(t), int(w-l), int(h-t)
        trackingResults.append([frameId, trackId, l, t, w, h])
        cv2.rectangle(frame, (l, t), (l+w, t+h), (0,255,0), 2)
        cv2.putText(frame, str(trackId), (l, t-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)

    videoWriter.write(frame)
    frameId += 1

cap.release()
videoWriter.release()

pd.DataFrame(trackingResults, columns=["frame", "id", "bbLeft", "bbTop", "bbWidth", "bbHeight"]).to_csv("task1_tracking.txt", index=False, header=False)


0: 384x640 19 persons, 1 umbrella, 3 handbags, 1 tv, 1066.6ms
Speed: 6.8ms preprocess, 1066.6ms inference, 14.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 18 persons, 2 umbrellas, 3 handbags, 1 tv, 988.4ms
Speed: 4.9ms preprocess, 988.4ms inference, 2.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 18 persons, 2 umbrellas, 4 handbags, 2 tvs, 925.8ms
Speed: 3.9ms preprocess, 925.8ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 17 persons, 1 backpack, 1 umbrella, 4 handbags, 1 tv, 925.2ms
Speed: 2.2ms preprocess, 925.2ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 14 persons, 1 backpack, 1 umbrella, 4 handbags, 1 tv, 995.6ms
Speed: 2.4ms preprocess, 995.6ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 15 persons, 1 backpack, 2 umbrellas, 4 handbags, 1273.4ms
Speed: 2.6ms preprocess, 1273.4ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)



In [20]:
# Cell 5: Load ground truth for Task1
gtColumns = ["frame", "id", "bbLeft", "bbTop", "bbWidth", "bbHeight", "conf", "cls", "visibility"]
gtData = pd.read_csv(task1GtPath, sep=",", header=None, names=gtColumns)
gtData = gtData.iloc[:, :6]

In [21]:
# Cell 6: Compute MOTA
def computeIou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[0]+boxA[2], boxB[0]+boxB[2])
    yB = min(boxA[1]+boxA[3], boxB[1]+boxB[3])
    interArea = max(0, xB-xA) * max(0, yB-yA)
    boxAArea = boxA[2]*boxA[3]
    boxBArea = boxB[2]*boxB[3]
    return interArea / float(boxAArea + boxBArea - interArea)

allFrames = sorted(gtData['frame'].unique())
fp, fn, idsw, gtCount = 0, 0, 0, 0
prevMatches = {}

for f in allFrames:
    gtFrame = gtData[gtData['frame']==f][["id","bbLeft","bbTop","bbWidth","bbHeight"]].astype(float).values
    predFrame = np.array([r[1:] for r in trackingResults if r[0]==f], dtype=float)

    gtCount += len(gtFrame)

    if len(gtFrame)==0 or len(predFrame)==0:
        fn += len(gtFrame)
        fp += len(predFrame)
        continue

    iouMatrix = np.zeros((len(gtFrame), len(predFrame)))
    for i, gtBox in enumerate(gtFrame):
        for j, predBox in enumerate(predFrame):
            iouMatrix[i,j] = computeIou(gtBox[1:], predBox[1:])

    gtIdx, predIdx = linear_sum_assignment(-iouMatrix)

    matchedGt, matchedPred = set(), set()
    for g, p in zip(gtIdx, predIdx):
        if iouMatrix[g,p] >= 0.5:
            matchedGt.add(g)
            matchedPred.add(p)
            if gtFrame[g,0] in prevMatches and prevMatches[gtFrame[g,0]] != predFrame[p,0]:
                idsw += 1
            prevMatches[gtFrame[g,0]] = predFrame[p,0]

    fn += len(gtFrame) - len(matchedGt)
    fp += len(predFrame) - len(matchedPred)

print("GT frames:", allFrames[:10])
print("Pred frames:", [r[0] for r in trackingResults[:10]])

mota = 1 - (fp + fn + idsw)/gtCount
print(f"MOTA: {mota:.4f}, FP: {fp}, FN: {fn}, IDSW: {idsw}, GT: {gtCount}")

GT frames: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]
Pred frames: [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
MOTA: 0.3335, FP: 2481, FN: 15049, IDSW: 231, GT: 26647


In [ ]:
# Cell 7: Task2 pedestrian counting
images = sorted([img for img in os.listdir(task2ImagesPath) if img.endswith(".jpg")], key=lambda x: int(x.split('.')[0]))
tracker = DeepSort(max_age=30, n_init=3)
counts = []

for frameId, imgName in enumerate(images, start=1):
    frame = cv2.imread(os.path.join(task2ImagesPath, imgName))
    results = yoloModel(frame)[0]
    boxes = results.boxes.xyxy.cpu().numpy()
    confidences = results.boxes.conf.cpu().numpy()
    classes = results.boxes.cls.cpu().numpy()

    detections = []
    for i, cls in enumerate(classes):
        if cls == 0:
            x1, y1, x2, y2 = boxes[i]
            detections.append(([x1, y1, x2-x1, y2-y1], confidences[i], "person"))

    tracks = tracker.update_tracks(detections, frame=frame)
    count = sum([1 for t in tracks if t.is_confirmed()])
    counts.append([frameId, count])

    for t in tracks:
        if not t.is_confirmed():
            continue
        l, t_, w, h = t.to_ltrb()
        l, t_, w, h = int(l), int(t_), int(w-l), int(h-t_)
        cv2.rectangle(frame, (l, t_), (l+w, t_+h), (0,255,0), 2)
        cv2.putText(frame, str(t.track_id), (l, t_-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)

    if frameId == 1:
        videoWriter = cv2.VideoWriter("task2.mp4", cv2.VideoWriter_fourcc(*'mp4v'), frameRate, (frame.shape[1], frame.shape[0]))
    videoWriter.write(frame)

videoWriter.release()
pd.DataFrame(counts, columns=["Number","Count"]).to_csv("task2_counts.csv", index=False)